In [4]:
#Not to be run

from email.message import EmailMessage
import os
from pathlib import Path
import random
import smtplib
import time
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

SENDER_EMAIL = os.getenv("GMAIL_USER2")
APP_PASSWORD = os.getenv("GMAIL_APP_PASSWORD2")

# Add contact names to personalize each email
# Path to your Excel file (r"" raw string handles backslashes safely)
#EXCEL_FILE_PATH = r"F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Gold\Email list\Email file.xlsx"


def load_recipients_from_excel(file_path):
    """Reads recipient names and emails from an Excel file."""
    excel_path = Path(file_path)

    if not excel_path.is_file():
        raise FileNotFoundError(f"Could not find Excel file at: {file_path}")

    # Read Excel sheet
    df = pd.read_excel(excel_path)

    # Clean data: drop rows where email or name is missing
    df = df.dropna(subset=["Project Proponent", "Email_1"])

    recipients = []
    for _, row in df.iterrows():
        # Convert values to clean strings
        name = str(row["Project Proponent"]).strip()
        email = str(row["Email_1"]).strip()

        # Simple validation to ensure valid email string
        if "@" in email:
            recipients.append({"name": name, "email": email})

    return recipients


# Automatically load RECIPIENTS from Excel
# RECIPIENTS = load_recipients_from_excel(EXCEL_FILE_PATH)

RECIPIENTS = [
    {
        "name": "Amit",
        "email": "amit_tayal2019@ampba.isb.edu"
    }
]

print(f"Successfully loaded {len(RECIPIENTS)} recipients from Excel.")

SUBJECT = (
    "RCC Chimney & Tall Structure Civil Construction Solutions |"
    " Nirmanshila Construction"
)

ATTACHMENT_PATH = r"F:\Chimney Work\Marketing\Brochure\Nirmanshila Construction - Brochure.pdf"


def get_html_body(name):
    """HTML version improves trust score with modern email providers."""
    return f"""\
<html>
  <body style="font-family: Arial, sans-serif; line-height: 1.6; color: #333333;">
    <p>Dear {name},</p>

    <p>I hope this email finds you well.</p>

    <p>I am writing to introduce <strong>Nirmanshila Construction</strong>. We are specialized in slipform constuction.</p>

    <p>If you are planning new expansions, modernizations, or structural maintenance, we offer end-to-end execution capabilities across the following core areas:</p>

    <ul>
      <li><strong>Tall Structural Construction:</strong> RCC Chimney, Silos, and Elevated Overhead Water Tanks.</li>
      <li><strong>Slipform Expertise:</strong> Experienced team for slipform work.</li>
      <li><strong>Thermal & Asset Protection:</strong> Refractory Brick Lining, Industrial Painting, and Protective Coatings.</li>
      <li><strong>RCC Chimney Repair/Maintenance and General Industrial Civil Works</strong>.</li>
    </ul>

    <p><strong>Why Partner with Nirmanshila?</strong></p>
    <ul>
      <li><strong>Technical Precision:</strong> From continuous-pour slipform setups to strict structural tolerances, we ensure flawless execution.</li>
      <li><strong>Operational Integrity:</strong> We pride ourselves on maintaining strict project timelines, safety standards, and transparent budgeting.</li>
      <li><strong>Turnkey Management:</strong> We handle the engineering complexities so your team can focus on core operations.</li>
    </ul>

    <p>For your review, I have attached our <strong>Company Brochure that includes company profile & project portfolio</strong>. You can also explore our capabilities online at <a href="https://www.nirmanshilaconstruction.com">www.nirmanshilaconstruction.com</a>.</p>

    <p>We would welcome the opportunity to connect to explore how we can support your upcoming projects. Let us know a good time to talk.</p>

    <p>Looking forward to hearing from you. Thanks!</p>

    <br>
    <p><strong>Rajeev Kumar</strong> &nbsp;|&nbsp; <strong>Amit Tayal</strong><br>
    +91 7500462001 &nbsp;|&nbsp; +91 9650744299</p>

    <hr style="border: none; border-top: 1px solid #cccccc; margin: 20px 0;">
    <p style="font-size: 0.9em; color: #555555;">
      <strong>M/s Nirmanshila Construction</strong><br>
      Registered Office: 17, Kushi Vihar, Shanti Nagar, Muzaffarnagar, 251001<br>
      Website: <a href="https://www.nirmanshilaconstruction.com">www.nirmanshilaconstruction.com</a><br>
      Email: info@nirmanshilaconstruction.com
    </p>
  </body>
</html>
"""


def create_email(recipient_data, pdf_data, pdf_name):
    msg = EmailMessage()
    msg["Subject"] = SUBJECT
    msg["From"] = f"Nirmanshila Construction <{SENDER_EMAIL}>"
    msg["To"] = recipient_data["email"]

    # Simple plain-text fallback
    plain_text = f"Dear {recipient_data['name']},\n\nPlease view this email in an HTML-compatible client."
    msg.set_content(plain_text)

    # Rich HTML content (Primary visual)
    msg.add_alternative(get_html_body(recipient_data["name"]), subtype="html")

    # Add attachment
    if pdf_data:
        msg.add_attachment(
            pdf_data,
            maintype="application",
            subtype="pdf",
            filename=pdf_name,
        )

    return msg


def send_bulk_emails_safely():
    pdf_file = Path(ATTACHMENT_PATH)
    pdf_data = pdf_file.read_bytes() if pdf_file.is_file() else None
    pdf_name = pdf_file.name if pdf_file.is_file() else None

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(SENDER_EMAIL, APP_PASSWORD)
            print("Connected to Gmail SMTP.")

            for idx, contact in enumerate(RECIPIENTS, start=1):
                msg = create_email(contact, pdf_data, pdf_name)
                server.send_message(msg)
                print(
                    f"[{idx}/{len(RECIPIENTS)}] Delivered to:"
                    f" {contact['email']}"
                )

                # Randomized 5 to 10-second delay between emails to appear human
                if idx < len(RECIPIENTS):
                    wait_time = random.randint(5, 10)
                    print(f"Waiting {wait_time}s before next send...")
                    time.sleep(wait_time)

        print("\nAll emails processed successfully!")

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    send_bulk_emails_safely()

Successfully loaded 1 recipients from Excel.
Connected to Gmail SMTP.
[1/1] Delivered to: amit_tayal2019@ampba.isb.edu

All emails processed successfully!
